# 🔤 Урок 17 — Токены, эмбеддинги, внимание (материалы преподавателя)

> 🎯 Цель: показать, как текст → числа, объяснить attention и природу галлюцинаций.

Блоки 1–3 работают офлайн. Блок 4 (BertViz) — в Colab с интернетом.

## Блок 1 · Токены
**Идея:** текст режется на кусочки, каждому — номер. Для модели фраза = список чисел.

In [ ]:
текст = 'кошка спит на диване'
словарь = {}; токены = []
for слово in текст.split():
    if слово not in словарь:
        словарь[слово] = len(словарь)   # новый номер
    токены.append(словарь[слово])
print('Слова :', текст.split())
print('Числа :', токены)   # для модели предложение — это список чисел

## Блок 2 · Эмбеддинги
**Идея:** каждый токен → вектор чисел. Близкие по смыслу слова — близкие векторы (высокая косинусная близость). Связь с уроком 14: там картинка = числа, здесь слова = числа.

In [ ]:
import numpy as np
emb = {
    'король':  np.array([0.9,0.8,0.1]),
    'королева':np.array([0.9,0.2,0.1]),
    'мужчина': np.array([0.8,0.9,0.2]),
    'женщина': np.array([0.8,0.1,0.2]),
    'банан':   np.array([0.1,0.4,0.9]),
}
def близость(a,b):
    a,b = emb[a],emb[b]
    return a@b/(np.linalg.norm(a)*np.linalg.norm(b))
print('король–королева:', round(близость('король','королева'),2))
print('король–банан   :', round(близость('король','банан'),2))

## Блок 3 · Внимание (иллюстрация)
**Идея:** предсказывая слово, модель смотрит на все слова и решает, на какие обращать внимание. Здесь 'она' смотрит на 'кошку'.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
слова = ['кошка','не','перешла','улицу','потому','что','она','устала']
attention = np.zeros((len(слова),len(слова)))
attention[6] = [0.6,0.02,0.05,0.05,0.03,0.03,0.1,0.12]   # 'она' -> 'кошка'
plt.imshow(attention, cmap='Purples')
plt.xticks(range(len(слова)), слова, rotation=45)
plt.yticks(range(len(слова)), слова)
plt.title('На какие слова смотрит каждое слово'); plt.colorbar(); plt.show()

## Блок 4 (в Colab) · BertViz на настоящей модели
Наведи на 'it' — увидишь связь со словом 'cat'.

In [ ]:
!pip install bertviz transformers -q
from bertviz import head_view
from transformers import AutoTokenizer, AutoModel
name='bert-base-uncased'
tok=AutoTokenizer.from_pretrained(name)
model=AutoModel.from_pretrained(name, output_attentions=True)
inputs=tok('The cat did not cross the street because it was tired', return_tensors='pt')
attn=model(**inputs).attentions
tokens=tok.convert_ids_to_tokens(inputs['input_ids'][0])
head_view(attn, tokens)

## Галлюцинации — связь с уроком 14
Модель выдаёт правдоподобное продолжение, а не истину. Как сеть из урока 14 уверенно называла цифру на шуме — LLM уверенно выдумывает факт. **Правдоподобно ≠ правда.**

---
**Итог.** Текст → токены → эмбеддинги → внимание. Модель не понимает — она считает числа и подбирает правдоподобное продолжение.